In [1]:
import pandas as pd
from sqlalchemy import create_engine

In [3]:


DATABASE_CONFIG = {
    'host': 'dandyweb01fl',
    'database': 'aedna_metadata_test',
    'port': '5432',
    'user': 'upload_user',
    'password': pw,
    'schema_name': 'test_1'
}

DATABASE_CONFIG = {
    'host': 'dandyweb01fl',
    'database': 'aedna_metadata_test',
    'port': '5432',
    'user': 'upload_user',
    'password': pw,
    'schema_name': 'test_1'
}

DATABASE_CONFIG_2 = {
    'host': DATABASE_CONFIG['host'],
    'dbname': DATABASE_CONFIG['database'],
    'port': DATABASE_CONFIG['port'],
    'user': DATABASE_CONFIG['user'],
    'password': DATABASE_CONFIG['password'],
}

DATABASE_CONFIG_READ_ONLY = {
    'host': DATABASE_CONFIG['host'],
    'dbname': DATABASE_CONFIG['database'],
    'port': DATABASE_CONFIG['port'],
    'user': 'read_user',
    'password': DATABASE_CONFIG['password'],
}

ENGINE = create_engine(
    f"postgresql://{DATABASE_CONFIG['user']}:{DATABASE_CONFIG['password']}@{DATABASE_CONFIG['host']}:{DATABASE_CONFIG['port']}/{DATABASE_CONFIG['database']}")

DATABASE_CONFIG_READ_ONLY = {
    'host': DATABASE_CONFIG['host'],
    'dbname': DATABASE_CONFIG['database'],
    'port': DATABASE_CONFIG['port'],
    'user': 'read_user',
    'password': DATABASE_CONFIG['password'],
}

ENGINE = create_engine(
    f"postgresql://{DATABASE_CONFIG_READ_ONLY['user']}:{DATABASE_CONFIG_READ_ONLY['password']}@{DATABASE_CONFIG_READ_ONLY['host']}:{DATABASE_CONFIG_READ_ONLY['port']}/{DATABASE_CONFIG_READ_ONLY['dbname']}")

In [ ]:
pd.set_option('display.max_rows', 1000)  # Set maximum number of rows to display
pd.set_option('display.max_columns', 1000)
wldf_q = "select * from test_1.edna_wetlab_report;"
asdf_q = "select * from test_1.edna_archive_sample;"
cgg_q = "select * from test_1.cgg_sediment_water;"
core_ids = ["ISL2020_05_25", "ISL2020_09_70", "ISL22_35.8", "ISL22_40.4", "ISL22_49.3b"]
wldf = pd.read_sql(wldf_q, dtype=str, con=ENGINE)
asdf = pd.read_sql(asdf_q, dtype=str, con=ENGINE)
cgg = pd.read_sql(cgg_q, dtype=str, con=ENGINE)

Cleaning the data so I can do proper filter and merges

In [107]:
# TODO: Make sure to check that replacemants doesnt result in duplicate data
core_ids = list(map(lambda x: x.upper(), core_ids))
core_ids = list(map(lambda x: x.replace(" ", ""), core_ids))
core_ids = list(map(lambda x: x.replace("," , "."), core_ids))
core_ids = list(map(lambda x: x.replace("-" , "_"), core_ids))
asdf["BulkSampleID"] = asdf["BulkSampleID"].str.strip()
asdf["BulkSampleID"] = asdf["BulkSampleID"].str.replace("-", "_")
asdf["BulkSampleID"] = asdf["BulkSampleID"].str.replace(",", ".")
asdf["BulkSampleID"] = asdf["BulkSampleID"].str.upper()
asdf["BulkSampleID"] = asdf["BulkSampleID"].str.replace(" ", "")
cgg["Museum ID/sample ID"] = cgg["Museum ID/sample ID"].str.strip()
cgg["Museum ID/sample ID"] = cgg["Museum ID/sample ID"].str.replace("," , ".")
cgg["Museum ID/sample ID"] = cgg["Museum ID/sample ID"].str.replace("-" , "_")
cgg["Museum ID/sample ID"] = cgg["Museum ID/sample ID"].str.upper()
cgg["Museum ID/sample ID"] = cgg["Museum ID/sample ID"].str.replace(" ", "")
cgg["CGG ID"] = cgg["CGG ID"].str.strip()
cgg["CGG ID"] = cgg["CGG ID"].str.replace("-" , "_")
cgg["CGG ID"] = cgg["CGG ID"].str.upper()
cgg["CGG ID"] = cgg["CGG ID"].str.replace(" ", "")
wldf["Archive Sample ID"] = wldf["Archive Sample ID"].str.strip()
wldf["Archive Sample ID"] = wldf["Archive Sample ID"].str.replace("-", "_")
wldf["Archive Sample ID"] = wldf["Archive Sample ID"].str.replace("," , ".")
wldf["Archive Sample ID"] = wldf["Archive Sample ID"].str.replace(" ", "")
wldf["Archive Sample ID"] = wldf["Archive Sample ID"].str.upper()

Getting all rows from cgg and jesper data where core id starts with the master core ids from Kurt

In [108]:
cores_kurt_cgg = cgg[cgg["Museum ID/sample ID"].isin(core_ids)]
cores_kurt_asdf = asdf[asdf["BulkSampleID"].isin(core_ids)]

Found 143 archive samples 

In [115]:
def find_duplicates(lst):
    seen = {}
    duplicates = []

    for item in lst:
        if item in seen:
            duplicates.append(item)
        else:
            seen[item] = 1

    return duplicates

Finding 1 duplicate core ID that is present in both Jespers and CGG:

In [116]:
my_list = list(cores_kurt_asdf["BulkSampleID"].unique()) + list(cores_kurt_cgg["Museum ID/sample ID"].unique())
find_duplicates(my_list)

[]

##### Merging
Merging on Archive Sample ID. Caution: Xihan says this ID is not robust in her data, so maybe this is error prone? Think it should be fine as long as its LV codes however. Best would be to merge with Robot Sample ID, will do so when I get the data from Jesper.

In [ ]:
merged_on_aID = pd.merge(cores_kurt_asdf, wldf, left_on='ArchiveSampleID', right_on='Archive Sample ID', how='left')
merged_on_aID_essentials = merged_on_aID[['BulkSampleID', "ArchiveSampleID", "Robot Sample ID", "Library ID", 'FastQ File ID', "DepthSampledCalTape"]]
len(merged_on_aID)
merged_on_aID_essentials.head()

Merging CGG number with Wetlab report

In [121]:
merged_on_CGG_ID = pd.merge(cores_kurt_cgg, wldf, left_on='CGG ID', right_on='Archive Sample ID', how='left')
merged_on_CGG_ID_essentials = merged_on_CGG_ID[["Museum ID/sample ID", 'CGG ID', "Library ID", 'FastQ File ID', "Depth", "height (m) asl.", "Age", "Geological age", "Lat", "Lon", "GPS"]]
len(merged_on_CGG_ID)

2

Merging Museum ID (Core segment id (field sample ID)) with wet lab report (shouldnt work)

In [ ]:
merged_on_museum_id = pd.merge(cores_kurt_cgg, wldf, left_on='Museum ID/sample ID', right_on='Archive Sample ID', how='left')
merged_on_museum_id_essentials = merged_on_museum_id[["Museum ID/sample ID", 'CGG ID', "Library ID", 'FastQ File ID', "Depth", "height (m) asl.", "Age", "Geological age", "Lat", "Lon", "GPS"]]
len(merged_on_museum_id)
merged_on_bulksampleid = pd.merge(cores_kurt_asdf, wldf, left_on='BulkSampleID', right_on='Archive Sample ID', how='left')
merged_on_bulksampleid_essentials = merged_on_bulksampleid[['BulkSampleID', "ArchiveSampleID", "Robot Sample ID", "Library ID", 'FastQ File ID', "DepthSampledCalTape"]]
len(merged_on_bulksampleid_essentials)
# Sometimes BulkSample IDs are CGG ids, which means we can get meta data from CGG database
def merge_back_to_cgg(concatenated_df):
    merged_df = pd.merge(concatenated_df, cgg, left_on="FieldSampleID", right_on="CGG ID", how='left')
    
    # Now replace NaN values in common columns of df1 with values from df2
    for column in concatenated_df.columns:
        if column in cgg.columns:
            merged_df[column+'_x'] = merged_df[column+'_x'].fillna(merged_df[column+'_y']).infer_objects(copy=False)
            merged_df = merged_df.drop(columns=[column+'_y'])

    # Rename the columns back to their original names
    full_merged_df = merged_df.rename(columns={col: col[:-2] for col in merged_df.columns if col.endswith('_x')})
cgg_essential = cgg[["Museum ID/sample ID", 'CGG ID', "Depth", "height (m) asl.", "Age", "Geological age", "Lat", "Lon", "GPS"]]
pd.set_option('future.no_silent_downcasting', True)
# Full 
merged_on_aID = merged_on_aID.rename(columns={"BulkSampleID": "FieldSampleID"})
merged_on_bulksampleid = merged_on_bulksampleid.rename(columns={"BulkSampleID": "FieldSampleID"})
merged_on_CGG_ID.loc[:, "FieldSampleID"] = merged_on_CGG_ID.loc[:, "Museum ID/sample ID"]
merged_on_museum_id.loc[:, "FieldSampleID"] = merged_on_museum_id.loc[:, "Museum ID/sample ID"]

conc = pd.concat([merged_on_aID, merged_on_bulksampleid, merged_on_CGG_ID, merged_on_museum_id])
conc = conc.reset_index()
merged_df = pd.merge(conc, cgg, left_on="FieldSampleID", right_on="CGG ID", how='left')
# Now replace NaN values in common columns of df1 with values from df2
for column in conc.columns:
    if column in cgg.columns:
        merged_df[column+'_x'] = merged_df[column+'_x'].fillna(merged_df[column+'_y']).infer_objects(copy=False)
        merged_df = merged_df.drop(columns=[column+'_y'])

# Rename the columns back to their original names
full_merged_df = merged_df.rename(columns={col: col[:-2] for col in merged_df.columns if col.endswith('_x')})

# Essential

# CLeaning
merged_on_aID_essentials = merged_on_aID_essentials.rename(columns={"BulkSampleID": "FieldSampleID"})
merged_on_bulksampleid_essentials = merged_on_bulksampleid_essentials.rename(columns={"BulkSampleID": "FieldSampleID"})
merged_on_CGG_ID_essentials.loc[:, "FieldSampleID"] = merged_on_CGG_ID_essentials.loc[:, "Museum ID/sample ID"]
merged_on_museum_id_essentials.loc[:, "FieldSampleID"] = merged_on_museum_id_essentials.loc[:, "Museum ID/sample ID"]

conc = pd.concat([merged_on_aID_essentials, merged_on_bulksampleid_essentials, merged_on_CGG_ID_essentials, merged_on_museum_id_essentials])
conc = conc.reset_index()

# Merging back into cgg to get the meta data from the cgg numbers that have been reported as field sample ids
merged_df = pd.merge(conc, cgg_essential, left_on="FieldSampleID", right_on="CGG ID", how='left')
for column in conc.columns:
    if column in cgg_essential.columns:
        merged_df[column+'_x'] = merged_df[column+'_x'].fillna(merged_df[column+'_y'])
        merged_df = merged_df.drop(columns=[column+'_y'])

# Rename the columns back to their original names
essential_merged_df = merged_df.rename(columns={col: col[:-2] for col in merged_df.columns if col.endswith('_x')})

In [144]:
essential_merged_df

,index,FieldSampleID,ArchiveSampleID,Robot Sample ID,Library ID,FastQ File ID,DepthSampledCalTape,Museum ID/sample ID,CGG ID,Depth,height (m) asl.,Age,Geological age,Lat,Lon,GPS
0,0,ISL22_40.4,LV3005273838,NaN,NaN,NaN,0.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,ISL22_40.4,LV3005272806,NaN,NaN,NaN,1.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,ISL22_40.4,LV3005273689,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,ISL22_40.4,LV3005272805,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,ISL22_40.4,LV3005272792,NaN,NaN,NaN,5.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,5,ISL22_40.4,LV3005272812,NaN,NaN,NaN,6.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,6,ISL22_40.4,LV3005272811,NaN,NaN,NaN,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,7,ISL22_40.4,LV3005272801,NaN,NaN,NaN,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,8,ISL22_40.4,LV3005273690,NaN,NaN,NaN,10.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,9,ISL22_40.4,LV3005273832,NaN,NaN,NaN,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
